# 01 YOLO11 Pose 영상 시각화

YOLO11 official pose pretrained weight로 사람 bbox와 skeleton을 영상에 그려 저장합니다. 기본값은 안전하게 아무 것도 실행하지 않습니다.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").is_file() and (path / "src").is_dir():
            return path
    raise RuntimeError("repo root를 찾지 못했습니다. notebook을 repo 안에서 실행하거나 REPO_ROOT를 직접 지정하세요.")


REPO_ROOT = find_repo_root(Path.cwd())
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("REPO_ROOT:", REPO_ROOT)

from labelstudio_bbox_tools.pose_inference.yolo11 import run_yolo11_pose_video_inference

## 설정

`INPUT_PATH`, `OUT_DIR`, `DEVICE`만 먼저 본인 환경에 맞춥니다. official pretrained weight는 `yolo11x-pose.pt`를 기본값으로 사용합니다.

In [ ]:
# ===== 사용자가 주로 바꾸는 설정 =====

# 영상 파일 하나 또는 영상 폴더를 넣습니다.
# 폴더를 넣으면 기본값(RECURSIVE=False) 기준으로 해당 폴더 바로 아래 영상만 찾습니다.
INPUT_PATH = Path("/path/to/video_or_video_folder")

# 결과 영상, predictions.jsonl, summary.json이 저장될 상위 폴더입니다.
OUT_DIR = Path("/path/to/pose_inference_outputs")

# YOLO official pose pretrained weight입니다.
# RF-DETR Keypoint Preview와 체급을 맞춘 비교가 목적이라 기본값은 가장 큰 official pose weight로 둡니다.
# 처음 실제 실행할 때 weight가 cache에 없으면 ultralytics가 다운로드를 시도할 수 있습니다.
MODEL_WEIGHTS = "yolo11x-pose.pt"

# official pose pretrained는 person 1-class라서 기본적으로 CLASS_YAML이 필요 없습니다.
# custom pose weight를 쓰는 경우에만 CLASS_YAML 또는 MANUAL_CLASSES를 채우세요.
CLASS_YAML = None
MANUAL_CLASSES = []

DEVICE = "cuda:0"       # CPU면 "cpu" 또는 None
IMGSZ = 640             # None, 640, 1280, 또는 (width, height)
CONF = 0.25             # instance/bbox confidence threshold
IOU = 0.45              # YOLO NMS IoU threshold
KEYPOINT_CONF = 0.20    # skeleton line/dot를 그릴 최소 keypoint confidence

RECURSIVE = False       # True로 바꾸면 하위 폴더까지 영상 탐색
MAX_VIDEOS = 1          # 처음엔 1개만 확인 추천. 전체 실행 시 None.
FRAME_STRIDE = 1        # 1이면 모든 frame 처리, 2면 2 frame마다 1개 처리
START_SECONDS = None    # 예: 10.0
END_SECONDS = None      # 예: 60.0
MAX_FRAMES = 300        # 처음엔 짧게 확인 추천. 전체 실행 시 None.

FONT_PATH = None        # None이면 Nanum/Noto CJK font를 자동 탐색합니다.
FONT_SIZE = 20
LINE_WIDTH = 3
KEYPOINT_RADIUS = 4
RUN_NAME = None         # None이면 시간 기반 폴더명 자동 생성
OVERWRITE = False

DRAW_BBOX = True
DRAW_SKELETON = True
DRAW_KEYPOINTS = True

# ===== 안전 플래그 =====
# 기본값에서는 아무 것도 실행하지 않습니다.
# 1) RUN_PREVIEW=True 로 입력 영상 목록/metadata만 확인
# 2) RUN_INFERENCE=True, DRY_RUN=False 로 실제 모델 로드 및 영상 저장
RUN_PREVIEW = False
RUN_INFERENCE = False
DRY_RUN = True

## 1. Preview

영상 목록과 metadata만 확인합니다. 모델을 load하지 않습니다.

In [ ]:
if RUN_PREVIEW:
    preview = run_yolo11_pose_video_inference(
        input_path=INPUT_PATH,
        out_dir=OUT_DIR,
        model_weights=MODEL_WEIGHTS,
        class_yaml=CLASS_YAML,
        manual_classes=MANUAL_CLASSES,
        device=DEVICE,
        imgsz=IMGSZ,
        conf=CONF,
        iou=IOU,
        keypoint_conf=KEYPOINT_CONF,
        recursive=RECURSIVE,
        max_videos=MAX_VIDEOS,
        frame_stride=FRAME_STRIDE,
        start_seconds=START_SECONDS,
        end_seconds=END_SECONDS,
        max_frames=MAX_FRAMES,
        run_name=RUN_NAME,
        font_path=FONT_PATH,
        font_size=FONT_SIZE,
        line_width=LINE_WIDTH,
        keypoint_radius=KEYPOINT_RADIUS,
        draw_bbox=DRAW_BBOX,
        draw_skeleton=DRAW_SKELETON,
        draw_keypoints=DRAW_KEYPOINTS,
        overwrite=OVERWRITE,
        dry_run=True,
    )
    preview.as_dict()
else:
    print("RUN_PREVIEW=False 이므로 preview를 건너뜁니다.")

## 2. Inference

실제 실행 전 `RUN_INFERENCE=True`, `DRY_RUN=False`로 바꿔야 모델을 load하고 결과 영상을 저장합니다.

In [ ]:
if RUN_INFERENCE:
    result = run_yolo11_pose_video_inference(
        input_path=INPUT_PATH,
        out_dir=OUT_DIR,
        model_weights=MODEL_WEIGHTS,
        class_yaml=CLASS_YAML,
        manual_classes=MANUAL_CLASSES,
        device=DEVICE,
        imgsz=IMGSZ,
        conf=CONF,
        iou=IOU,
        keypoint_conf=KEYPOINT_CONF,
        recursive=RECURSIVE,
        max_videos=MAX_VIDEOS,
        frame_stride=FRAME_STRIDE,
        start_seconds=START_SECONDS,
        end_seconds=END_SECONDS,
        max_frames=MAX_FRAMES,
        run_name=RUN_NAME,
        font_path=FONT_PATH,
        font_size=FONT_SIZE,
        line_width=LINE_WIDTH,
        keypoint_radius=KEYPOINT_RADIUS,
        draw_bbox=DRAW_BBOX,
        draw_skeleton=DRAW_SKELETON,
        draw_keypoints=DRAW_KEYPOINTS,
        overwrite=OVERWRITE,
        dry_run=DRY_RUN,
    )
    result.as_dict()
else:
    print("RUN_INFERENCE=False 이므로 실제 inference를 건너뜁니다.")